# Module 6: Machine Models — The M1 Stack Machine

**Prerequisites:** Modules 1–5 (Introduction, Logic & Proofs, Recursion & Induction, Data Structures, Graph Algorithms)

In this module, we study how to *model a computer* inside a theorem prover and *prove that programs are correct*. We use the M1 machine — a simplified JVM-like stack machine designed by J Strother Moore for teaching program verification with ACL2. The M1 machine is defined in `books/demos/marktoberdorf-08/m1.lisp`.

## 1. Why Machine Models?

Software verification ultimately requires a *formal model of computation*. Without one, we can prove properties of mathematical functions, but not that an *actual program* computes the right answer.

A machine model gives us:
- A precise definition of what each instruction does
- A state that we can reason about formally
- A `step` function that describes one execution step
- A `run` function that executes multiple steps

### The Verification Pipeline

$$\text{Specification (math function)} \xleftarrow{\text{prove } =} \text{Program on Machine Model} \xrightarrow{\text{compile}} \text{Real Hardware}$$

## 2. The M1 Machine

M1 is a tiny JVM-like stack machine with:
- A **program counter** (pc): index into the instruction list
- **Local variables**: a list of values, accessed by index
- An **operand stack**: used for arithmetic and comparisons
- A **program**: a list of instructions

### Instruction Set

| Instruction | Effect | Stack |
|-------------|--------|-------|
| `(ICONST k)` | Push constant $k$ | $\ldots \Rightarrow \ldots, k$ |
| `(ILOAD i)` | Push `locals[i]` | $\ldots \Rightarrow \ldots, v$ |
| `(ISTORE i)` | Pop into `locals[i]` | $\ldots, v \Rightarrow \ldots$ |
| `(IADD)` | Pop two, push sum | $\ldots, a, b \Rightarrow \ldots, a+b$ |
| `(ISUB)` | Pop two, push difference | $\ldots, a, b \Rightarrow \ldots, a-b$ |
| `(IMUL)` | Pop two, push product | $\ldots, a, b \Rightarrow \ldots, a \times b$ |
| `(IFEQ off)` | Pop; if 0, jump | $\ldots, v \Rightarrow \ldots$ |
| `(IFLE off)` | Pop; if $\leq 0$, jump | $\ldots, v \Rightarrow \ldots$ |
| `(IFLT off)` | Pop; if $< 0$, jump | $\ldots, v \Rightarrow \ldots$ |
| `(GOTO off)` | Unconditional jump | (unchanged) |
| `(HALT)` | Stop execution | (unchanged) |

## 3. Defining the Machine

### State Representation and Accessors

In [ ]:
; M1 State constructor and accessors
(defun make-state (pc locals stack program)
  (list pc locals stack program))

(defun pc (s) (nth 0 s))
(defun locals (s) (nth 1 s))
(defun stack (s) (nth 2 s))
(defun program (s) (nth 3 s))

In [ ]:
; Stack operations for M1
(defun push-stack (v stk) (cons v stk))
(defun top-stack (stk) (car stk))
(defun pop-stack (stk) (cdr stk))

; Instruction fetch and decode
(defun current-instruction (s)
  (nth (pc s) (program s)))

(defun opcode (inst) (car inst))
(defun arg1 (inst) (cadr inst))

; Local variable access
(defun local-var (i locals)
  (nth i locals))

(defun update-local (i v locals)
  (update-nth i v locals))

### The Step Function

The heart of M1 — executes one instruction and returns the new state:

In [ ]:
; Execute one instruction
(defun execute-instruction (inst s)
  (let ((op (opcode inst)))
    (case op
      (ICONST
       (make-state (+ 1 (pc s))
                   (locals s)
                   (push-stack (arg1 inst) (stack s))
                   (program s)))
      (ILOAD
       (make-state (+ 1 (pc s))
                   (locals s)
                   (push-stack (local-var (arg1 inst) (locals s))
                               (stack s))
                   (program s)))
      (ISTORE
       (make-state (+ 1 (pc s))
                   (update-local (arg1 inst)
                                 (top-stack (stack s))
                                 (locals s))
                   (pop-stack (stack s))
                   (program s)))
      (IADD
       (make-state (+ 1 (pc s))
                   (locals s)
                   (push-stack (+ (top-stack (pop-stack (stack s)))
                                  (top-stack (stack s)))
                               (pop-stack (pop-stack (stack s))))
                   (program s)))
      (ISUB
       (make-state (+ 1 (pc s))
                   (locals s)
                   (push-stack (- (top-stack (pop-stack (stack s)))
                                  (top-stack (stack s)))
                               (pop-stack (pop-stack (stack s))))
                   (program s)))
      (IMUL
       (make-state (+ 1 (pc s))
                   (locals s)
                   (push-stack (* (top-stack (pop-stack (stack s)))
                                  (top-stack (stack s)))
                               (pop-stack (pop-stack (stack s))))
                   (program s)))
      (IFEQ
       (make-state (if (equal (top-stack (stack s)) 0)
                       (+ (pc s) (arg1 inst))
                     (+ 1 (pc s)))
                   (locals s)
                   (pop-stack (stack s))
                   (program s)))
      (IFLE
       (make-state (if (<= (top-stack (stack s)) 0)
                       (+ (pc s) (arg1 inst))
                     (+ 1 (pc s)))
                   (locals s)
                   (pop-stack (stack s))
                   (program s)))
      (IFLT
       (make-state (if (< (top-stack (stack s)) 0)
                       (+ (pc s) (arg1 inst))
                     (+ 1 (pc s)))
                   (locals s)
                   (pop-stack (stack s))
                   (program s)))
      (GOTO
       (make-state (+ (pc s) (arg1 inst))
                   (locals s)
                   (stack s)
                   (program s)))
      (HALT s)
      (otherwise s))))

In [ ]:
; Take one step (if not halted)
(defun m1-step (s)
  (let ((inst (current-instruction s)))
    (if (equal (opcode inst) 'HALT)
        s
      (execute-instruction inst s))))

; Run for n steps (the 'clock')
(defun run (s n)
  (if (zp n)
      s
    (run (m1-step s) (- n 1))))

### Testing the Machine

In [ ]:
; Simple program: compute 3 + 4
(defconst *add-program*
  '((ICONST 3)
    (ICONST 4)
    (IADD)
    (HALT)))

; Run 3 steps and check the result
(let ((result (run (make-state 0 nil nil *add-program*) 3)))
  (top-stack (stack result)))  ; should be 7

## 4. Example: Factorial Program

### Mathematical Specification

In [ ]:
; Mathematical factorial
(defun fact (n)
  (if (zp n)
      1
    (* n (fact (- n 1)))))

(list (fact 0) (fact 1) (fact 5) (fact 10))

### Factorial as M1 Bytecode

Local variables: `locals[0]` = n (counts down), `locals[1]` = acc (starts at 1).

Algorithm: `acc = 1; while (n > 0) { acc = acc * n; n = n - 1; }`

```
pc  Instruction       Comment
--  -----------       -------
 0  (ILOAD 0)         push n
 1  (IFEQ 10)         if n == 0, jump to pc 11
 2  (ILOAD 1)         push acc
 3  (ILOAD 0)         push n
 4  (IMUL)            acc * n
 5  (ISTORE 1)        store into acc
 6  (ILOAD 0)         push n
 7  (ICONST 1)        push 1
 8  (ISUB)            n - 1
 9  (ISTORE 0)        store into n
10  (GOTO -10)        jump back to pc 0
11  (ILOAD 1)         push result
12  (HALT)
```

In [ ]:
; Factorial program for M1
(defconst *factorial-program*
  '((ILOAD 0)          ; 0: push n
    (IFEQ 10)          ; 1: if n=0 goto 11
    (ILOAD 1)          ; 2: push acc
    (ILOAD 0)          ; 3: push n
    (IMUL)             ; 4: acc*n
    (ISTORE 1)         ; 5: acc = acc*n
    (ILOAD 0)          ; 6: push n
    (ICONST 1)         ; 7: push 1
    (ISUB)             ; 8: n-1
    (ISTORE 0)         ; 9: n = n-1
    (GOTO -10)         ; 10: goto 0
    (ILOAD 1)          ; 11: push result
    (HALT)))           ; 12: stop

In [ ]:
; Run factorial(5): initial state pc=0, locals=(5 1), stack=nil
(defconst *fact-5-result*
  (run (make-state 0 '(5 1) nil *factorial-program*) 100))

; The answer should be 120 = 5!
(list (top-stack (stack *fact-5-result*))
      (fact 5)
      (equal (top-stack (stack *fact-5-result*)) (fact 5)))

### Step-by-Step Execution

In [ ]:
; Trace the first few steps for factorial(3)
(let ((s0 (make-state 0 '(3 1) nil *factorial-program*)))
  (list
    (list 'step-0 s0)
    (list 'step-1 (run s0 1))    ; ILOAD 0: push 3
    (list 'step-2 (run s0 2))    ; IFEQ: 3≠0, continue
    (list 'step-3 (run s0 3))    ; ILOAD 1: push acc=1
    (list 'step-4 (run s0 4))    ; ILOAD 0: push n=3
    (list 'step-5 (run s0 5))    ; IMUL: 1*3=3
    (list 'step-6 (run s0 6))))  ; ISTORE 1: acc=3

## 5. Example: Iterative Sum

Compute $\sum_{i=1}^{n} i = \frac{n(n+1)}{2}$.

In [ ]:
; Mathematical sum from 1 to n
(defun sum-1-to-n (n)
  (if (zp n)
      0
    (+ n (sum-1-to-n (- n 1)))))

; The closed form
(defthm sum-closed-form
  (implies (natp n)
           (equal (sum-1-to-n n)
                  (/ (* n (+ n 1)) 2))))

In [ ]:
; Sum program: locals[0]=n (down-counter), locals[1]=sum (accumulator)
(defconst *sum-program*
  '((ILOAD 0)          ; 0: push n
    (IFEQ 10)          ; 1: if n=0 goto 11
    (ILOAD 1)          ; 2: push sum
    (ILOAD 0)          ; 3: push n
    (IADD)             ; 4: sum + n
    (ISTORE 1)         ; 5: sum = sum + n
    (ILOAD 0)          ; 6: push n
    (ICONST 1)         ; 7: push 1
    (ISUB)             ; 8: n - 1
    (ISTORE 0)         ; 9: n = n - 1
    (GOTO -10)         ; 10: goto 0
    (ILOAD 1)          ; 11: push result
    (HALT)))           ; 12: stop

; Run sum(10) and verify
(let ((result (run (make-state 0 '(10 0) nil *sum-program*) 200)))
  (list (top-stack (stack result))
        (sum-1-to-n 10)
        (/ (* 10 11) 2)))

### Sum Program: Step-by-Step Verification

Let's verify the sum program computes $\frac{n(n+1)}{2}$:

In [ ]:
; Clock function for the sum program
(defun sum-clock (n)
  (if (zp n)
      2
    (+ 11 (sum-clock (- n 1)))))

; The M1 sum function
(defun m1-sum (n)
  (top-stack
    (stack
      (run (make-state 0 (list n 0) nil *sum-program*)
           (sum-clock n)))))

; Test
(list (equal (m1-sum 10) (sum-1-to-n 10))
      (equal (m1-sum 100) (sum-1-to-n 100)))

### Understanding the Clock

The clock function reflects the program's control flow:
- If $n = 0$: we execute 2 instructions (test $n$, load result + halt)
- Each iteration: 11 instructions (test, load, load, add, store, load, push, sub, store, goto, then re-test)
- Total for input $n$: $11n + 2$ steps

The clock function is a *constructive termination argument* — it proves the program terminates by giving an exact step count.

In [ ]:
; Verify the clock is linear
(list (fact-clock 0)   ; 2
      (fact-clock 1)   ; 13
      (fact-clock 5)   ; 57
      (fact-clock 10)) ; 112  = 11*10 + 2

### Running Multiple Programs

We can define additional programs and verify them using the same methodology. The key insight is that the machine definition is *fixed* — only the program and initial state change:

In [ ]:
; A program to compute the absolute value of locals[0]
(defconst *abs-program*
  '((ILOAD 0)       ; 0: push n
    (IFLT 4)        ; 1: if n < 0, jump to pc 5
    (ILOAD 0)       ; 2: push n (already non-negative)
    (GOTO 5)        ; 3: jump to halt
    (ICONST 0)      ; 4: (unreachable with IFLT jumping +4)
    (ICONST 0)      ; 5: push 0
    (ILOAD 0)       ; 6: push n
    (ISUB)           ; 7: 0 - n = -n
    (HALT)))         ; 8: stop

; Test: abs(-7) = 7, abs(5) = 5
(list (top-stack (stack (run (make-state 0 '(-7) nil *abs-program*) 5)))
      (top-stack (stack (run (make-state 0 '(5) nil *abs-program*) 5)))

## 6. Clock Functions

The key proof technique for M1 programs is the *clock function*. A clock function tells us exactly how many steps a program needs to reach completion.

To prove `(run state n)` produces the right answer, we need:
1. The right number of steps `n` (the *clock*)
2. A proof that after `n` steps, the result is correct

In [ ]:
; Clock function for the factorial program
; Each loop iteration takes 11 steps, final exit takes 2
(defun fact-clock (n)
  (if (zp n)
      2  ; test n=0, load result, halt
    (+ 11 (fact-clock (- n 1)))))

; How many steps for factorial(5)?
(fact-clock 5)

In [ ]:
; The M1 factorial function: run with the correct clock
(defun m1-factorial (n)
  (top-stack
    (stack
      (run (make-state 0 (list n 1) nil *factorial-program*)
           (fact-clock n)))))

; Test that m1-factorial matches fact
(list (equal (m1-factorial 0) (fact 0))
      (equal (m1-factorial 1) (fact 1))
      (equal (m1-factorial 5) (fact 5))
      (equal (m1-factorial 10) (fact 10)))

### The Correctness Theorem

$$\forall n \in \mathbb{N}: \text{m1-factorial}(n) = n!$$

In [ ]:
; M1 factorial equals mathematical factorial
(defthm m1-factorial-is-fact
  (implies (natp n)
           (equal (m1-factorial n)
                  (fact n))))

## 7. Loop Invariants in M1

### The Classical Approach

Proving a loop correct requires a *loop invariant* — a property that holds at each iteration:
1. **Initialization**: holds when the loop starts
2. **Maintenance**: preserved by each iteration
3. **Termination**: combined with exit condition, implies the postcondition

This is the Hoare logic approach: $\{\text{Inv} \land \text{condition}\} \;\text{body}\; \{\text{Inv}\}$

### Factorial Loop Invariant

At pc=0: $\text{acc} \times n! = N!$ where $N$ is the original input.

- **Init**: $\text{acc} = 1, n = N \Rightarrow 1 \times N! = N!$ ✓
- **Maintenance**: $\text{acc}' = \text{acc} \times n, n' = n-1 \Rightarrow (\text{acc} \times n) \times (n-1)! = \text{acc} \times n! = N!$ ✓
- **Termination**: $n = 0 \Rightarrow \text{acc} \times 0! = \text{acc} = N!$ ✓

In [ ]:
; Loop invariant: acc * fact(n) = fact(N)
(defun fact-loop-invariant (n acc)
  (* acc (fact n)))

; The invariant is preserved by one iteration
(defthm fact-invariant-maintained
  (implies (and (natp n) (not (zp n)))
           (equal (fact-loop-invariant (- n 1) (* acc n))
                  (fact-loop-invariant n acc))))

; At termination (n=0), the invariant gives the answer
(defthm fact-invariant-at-exit
  (equal (fact-loop-invariant 0 acc)
         acc))

### Connection to Hoare Logic

| Hoare Logic | M1 Approach |
|---|---|
| Precondition $\{P\}$ | Initial state conditions |
| Postcondition $\{Q\}$ | Properties of final state |
| Loop invariant | Invariant at loop head (pc=0) |
| Weakest precondition | Clock function derivation |
| Proof rule for loops | Induction on clock |

The advantage of M1: everything is mechanically checked by ACL2 — no gaps in reasoning.

### Verification Methodology Summary

1. **Write the program** as M1 bytecode
2. **Define the specification** as a mathematical function
3. **Define the clock function** computing steps needed
4. **State the loop invariant** relating machine state to specification
5. **Prove the invariant is maintained** through each iteration
6. **Prove the final theorem**: running with the clock gives the specification

## 8. Exercises

### Exercise 6.1: Multiplication by Repeated Addition

Write an M1 program that computes $a \times b$ using repeated addition (no IMUL). Use locals[0]=a, locals[1]=b (counts down), locals[2]=result. Define the clock function and prove correctness.

In [ ]:
; Exercise 6.1: Multiplication by repeated addition
; YOUR CODE HERE


### Exercise 6.2: Fibonacci

Write an M1 program that computes the $n$-th Fibonacci number. Use locals[0]=n (counter), locals[1]=fib(k), locals[2]=fib(k-1). Prove your program matches the mathematical Fibonacci.

In [ ]:
; Exercise 6.2: Fibonacci on M1
; YOUR CODE HERE


### Exercise 6.3: Exponentiation

Write an M1 program that computes $b^n$ using repeated multiplication. Define the clock function and prove correctness.

*Bonus:* Write a fast-exponentiation version using squaring. What does the clock function look like?

In [ ]:
; Exercise 6.3: Exponentiation on M1
; YOUR CODE HERE


### Exercise 6.4: Verify Your Own Program

Choose a simple algorithm (GCD, integer division, counting) and:
1. Write it as M1 bytecode
2. Define its mathematical specification
3. Define the clock function
4. State and prove the loop invariant
5. Prove the correctness theorem

In [ ]:
; Exercise 6.4: Your own M1 program
; YOUR CODE HERE


## Summary

In this module, we learned:

- **Machine models** provide a formal foundation for program verification
- **M1** is a simple JVM-like stack machine with 11 instructions
- The **state** is `(pc, locals, stack, program)`
- **Clock functions** determine how many steps a program needs
- **Loop invariants** are the key technique for verifying loops
- The methodology connects directly to **Hoare logic**

| Module | What We Verified |
|--------|------------------|
| 2–3 | Mathematical functions (factorial, append, etc.) |
| 4 | Data structure operations (BST insert, lookup) |
| 5 | Graph algorithms (DFS, Dijkstra) |
| **6** | **Programs running on a machine model** |

$$\text{Math} \rightarrow \text{Data Structures} \rightarrow \text{Algorithms} \rightarrow \text{Programs}$$

---

**Congratulations!** You have completed the core modules of the ACL2 CS Tutorial. You now have the foundations to explore:

- **Compiler verification**: Proving a compiler correctly translates to M1 bytecode
- **Operating system verification**: Modeling process scheduling and memory management
- **Hardware verification**: Using ACL2 to verify processor designs (as done at AMD)
- **Cryptographic verification**: Proving properties of cryptographic algorithms

The ACL2 community books contain thousands of verified programs, algorithms, and machine models to learn from.